In [6]:
# for the MACE MD using ASE ######

import ase.io
from ase.calculators.espresso import Espresso
import os
import numpy as np


sample_name = 'GO-20-1-npt' # system name 
traj_path = "../4-mace-finetune/mace-training-12/md/3"
folder = "../4-mace-finetune/mace-training-12/md/20-1-/20-1-npt/"
out_dir = os.path.join(folder, f"extracted-config_min-{sample_name}")
os.makedirs(out_dir, exist_ok=True)

n_structures = 10 # number of structures we need to perform DFT calulation
skip_fraction = 0.0

# =============================================================================================
#  QE & pseudopotentials                                        
# =============================================================================================
pseudopotentials = {
    'H': 'H.pbe-rrkjus_psl.1.0.0.UPF', 
    'C': 'C.pbe-n-rrkjus_psl.1.0.0.UPF',
    'O': 'O.pbe-n-rrkjus_psl.1.0.0.UPF',
}
    'pseudo_dir': '/home/sara/qe/pseudo',
   # 'pseudo_dir': '/home/cicn/cicn864958/qe/pseudo',
input_qe_std = {
    'calculation': 'scf',
    'outdir': '.',
    'pseudo_dir': '/home/sara/qe/pseudo',
    'disk_io': 'none',
    'system': {
        'ecutwfc': 90,
        'nosym': True,
        'input_dft': 'PBE',
        'occupations': 'smearing',
        'smearing': 'fermi-dirac',
        'degauss': 0.001,
        'ibrav': 0,
        'nspin': 1,
    },
    'electrons': {
        'mixing_beta': 0.3,
        'electron_maxstep': 1000,
        'mixing_mode': 'plain',
        'conv_thr': 1e-8,
        'diagonalization': 'david',
    },
}



traj = ase.io.read(traj_path, format="extxyz", index=":")
n_frames = len(traj)
print(f"Loaded {n_frames} frames from NPT trajectory")

if n_frames == 0:
    raise RuntimeError("No frames found in trajectory")

# =====================================================
# UNIFORM SAMPLING

start = int(skip_fraction * n_frames)
if start >= n_frames:
    start = 0

if n_structures > (n_frames - start):
    raise ValueError("Requested more structures than available frames")

selected_indices = np.linspace(
    start, n_frames - 1, n_structures, dtype=int
)

print("Selected frame indices:", selected_indices)

# =====================================================
# WRITE QE INPUTS

for i, fi in enumerate(selected_indices, start=1):
    atoms = traj[fi]

    fname = os.path.join(
        out_dir, f"pw-{sample_name}-{i:03d}.in"
    )

    ase.io.write(
        fname,
        atoms,
        format="espresso-in",
        input_data=input_qe_std,
        pseudopotentials=pseudopotentials,
        tprnfor=True
    )

    print(f"→ wrote {fname} (frame {fi})")

print(f"\n Extraction done! Total QE inputs written: {n_structures}")